# Example 6.3 / 1: Branching on Solve Status

Companion notebook for **6.3 if-then-else in Scripts** in the AMPL lab Documentation, section "Branching on Solve Status". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Branching on Solve Status

This pattern reads solution values only when AMPL reports the expected general result. In the
Python tabs the branch is Python's own `if` on `ampl.solve_result`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
param avail >= 0;                         # hours available
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} Make[p] / rate[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs")
if ampl.solve_result == "solved":
    print(f"objective = {ampl.obj['Total_Profit'].value():.2f}")
else:
    print(f"solve failed: {ampl.solve_result}")

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
objective = 192000.00


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; here the AMPL `if` command makes the choice:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param: rate profit market :=
  bands 200 25 6000
  coils 140 30 4000;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
if solve_result = "solved" then {
    printf "objective = %.2f\n", Total_Profit;
} else {
    printf "solve failed: %s\n", solve_result;
}

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
objective = 192000.00
